# 02 - Features, training, saving `models/model.pkl`
How it works: an **Isolation Forest** is trained on NORMAL rows only. Its raw score is calibrated so that **0.5 = the 99.9th percentile of held-out normal data**. Simple rules then name the fault type.
The same `build_features()` is used here and in `score_reading()`, which is the contract's features rule.

In [ ]:
import pandas as pd, numpy as np

from pathlib import Path
DATA_CANDIDATES = [Path("../data/sim_30d.csv"), Path("../data/synthetic_30d.csv")]
DATA = next((p for p in DATA_CANDIDATES if p.exists()), None)
print("using:", DATA)

from energyguard_ml.training import load_csv, time_split, fit_artifact, save_artifact
THRESH_PCT = 99.9      # tune: 99.5 (more alerts) ... 99.95 (fewer alerts)

## Chronological split: first 70% of time = train, last 30% = test (test is only used in notebook 03)

In [ ]:
df = load_csv(DATA)
train, test = time_split(df, 0.7)
print("train rows:", len(train), " test rows:", len(test))
print("normal rows in train:", int((train.sim_label == "normal").sum()))
# If you also have a 5-second-cadence CSV, load it the same way and concatenate the
# TRAIN parts: pd.concat([train, train_5s]). The 'source' column keeps cadences apart.

## Train (builds features with build_features, fits the forest, calibrates, learns rule thresholds). Takes about a minute for 130k rows.

In [ ]:
import time
t0 = time.time()
artifact = fit_artifact(train, threshold_pct=THRESH_PCT)
print("trained in %.0fs on %d normal rows" % (time.time() - t0, artifact["n_train_normal"]))
print("calibration:", artifact["calibration"])
th = artifact["thresholds"]
print("spike_w=%.0f  z_hi=%.1f  v band=[%.1f, %.1f]" % (th["spike_w"], th["z_hi"], th["v_lo"], th["v_hi"]))
print("quiet (low) UTC hours:", th["low_hours"])

In [ ]:
path = save_artifact(artifact, "../models/model.pkl")
print("saved:", path.resolve(), "-", round(path.stat().st_size / 1e6, 2), "MB")
import sklearn; print("scikit-learn used:", sklearn.__version__, "(must equal the version pinned in pyproject.toml)")